# Task 3: Sequence Classification

## Long Short-Term Memory (LSTM)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)


### Load data

In [3]:
X_train_w = np.load('../data/3_processed/X_train_w.npy')
y_train_w = np.load('../data/3_processed/y_train_w.npy')
X_test_w = np.load('../data/3_processed/X_test_w.npy')
y_test_w = np.load('../data/3_processed/y_test_w.npy')

### Encode Labels

In PyTorch, the `CrossEntropyLoss` function for training the model does not accept text ("N", "S", "V"). It requires that the target labels be integer indices, starting from 0.

In [4]:
to_encode = {"N": 0, "S": 1, "V": 2}
if y_train_w.dtype.kind in {'U', 'S', 'O'}:
    y_train_w = np.vectorize(to_encode.get)(y_train_w)
if y_test_w.dtype.kind in {'U', 'S', 'O'}:
    y_test_w = np.vectorize(to_encode.get)(y_test_w)
    
print(f"Unique train labels: {np.unique(y_train_w)}")
print(f"Unique test labels: {np.unique(y_test_w)}")

Unique train labels: [0 1 2]
Unique test labels: [0 1 2]


### PyTorch Tensors

`y` contains the classification labels. After mapping, these are integers with no decimals and represent discrete categories. Thes numbers are not mathematical values; they are indices (`torch.long`).

In [5]:
X_train_tensor = torch.tensor(X_train_w, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_w, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_w, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_w, dtype=torch.long)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")


X_train_tensor shape: torch.Size([24027, 10, 8])
y_train_tensor shape: torch.Size([24027])


### PyTorch Dataset and DataLoader

In [6]:
class ECGSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = ECGSequenceDataset(X_train_tensor, y_train_tensor)
test_dataset = ECGSequenceDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

A `batch_size` of 64 provides an average that is good enough to point the model in the right direction, but it retains a small amount of noise from one batch to the next. This noise is beneficial because it acts as a natural regularizer and helps the network generalize better to the test data.

The training tensor has  24,027 training examples:
$$
\frac{24,027}{64} \approx 375
$$
This means that in each training epoch, the model will adjust its weights 375 times.

### LSTM Architecture Definition

In [7]:
class LSTMClassifier(nn.Module):
    def __init__(
            self,
            input_size=8,
            hidden_size=64,
            num_layers=1,
            num_classes=3
    ):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        lstm_output, (hidden, cell) = self.lstm(x)
        last_output = lstm_output[:, -1, :]  # Take the last time step's output
        output = self.fc(last_output)
        return output

### Initializing Model, Optimizer and Loss

In [15]:
# Check for GPU availability
# 'cuda' is the common name for NVIDIA GPUs in PyTorch
# 'cpu' is the fallback if no GPU is available 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model_lstm = LSTMClassifier(
    input_size=8,
    hidden_size=64,
    num_layers=1,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

num_epochs = 20

Using device: cpu


### Classifier Training and Validation

In [ ]:
def process_epoch(model, dataloader, criterion, optimizer=None):
    """
    Unified function for processing an epoch.
    If optimizer is provided, it trains. Otherwise, it evaluates.
    """
    is_training = optimizer is not None

    # Model mode
    model_lstm.train() if is_training else model_lstm.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_training): # Enable gradients only during training
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device) # Move input to device
            y_batch = y_batch.to(device) # Move labels to device

            outputs = model_lstm(X_batch) # Forward pass
            loss = criterion(outputs, y_batch) # Compute loss

            # Backpropagation and optimization only during training
            if is_training:
                optimizer.zero_grad() # Clear gradients
                loss.backward() # Backpropagate loss
                optimizer.step() # Update model parameters

            running_loss += loss.item() * X_batch.size(0) # Accumulate loss
            _, predicted = torch.max(outputs.data, 1) # Get predicted labels
            total += y_batch.size(0) # Update total samples
            correct += (predicted == y_batch).sum().item() # Update correct predictions

    epoch_loss = running_loss / total # Average loss over the epoch
    epoch_acc = correct / total # Accuracy for the epoch
    return epoch_loss, epoch_acc


train_losses = []
test_losses = []
train_acc = []
test_acc = []

for epoch in range(num_epochs):
    train_loss, train_accuracy = process_epoch(model_lstm, train_loader, criterion, optimizer)
    test_loss, test_accuracy = process_epoch(model_lstm, test_loader, criterion)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_acc.append(train_accuracy)
    test_acc.append(test_accuracy)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.4f}"
    )

Epoch [1/20] Train Loss: 0.1844, Train Acc: 0.9442 | Test Loss: 0.0617, Test Acc: 0.9822
Epoch [2/20] Train Loss: 0.0520, Train Acc: 0.9851 | Test Loss: 0.0421, Test Acc: 0.9860
Epoch [3/20] Train Loss: 0.0361, Train Acc: 0.9897 | Test Loss: 0.0324, Test Acc: 0.9895
Epoch [4/20] Train Loss: 0.0272, Train Acc: 0.9924 | Test Loss: 0.0254, Test Acc: 0.9927
Epoch [5/20] Train Loss: 0.0214, Train Acc: 0.9945 | Test Loss: 0.0219, Test Acc: 0.9942
Epoch [6/20] Train Loss: 0.0172, Train Acc: 0.9958 | Test Loss: 0.0227, Test Acc: 0.9937
Epoch [7/20] Train Loss: 0.0140, Train Acc: 0.9967 | Test Loss: 0.0186, Test Acc: 0.9955
Epoch [8/20] Train Loss: 0.0118, Train Acc: 0.9973 | Test Loss: 0.0199, Test Acc: 0.9947
Epoch [9/20] Train Loss: 0.0104, Train Acc: 0.9979 | Test Loss: 0.0200, Test Acc: 0.9947
Epoch [10/20] Train Loss: 0.0095, Train Acc: 0.9980 | Test Loss: 0.0175, Test Acc: 0.9957
Epoch [11/20] Train Loss: 0.0085, Train Acc: 0.9983 | Test Loss: 0.0180, Test Acc: 0.9950
Epoch [12/20] Train